Modelo

In [ ]:
from models.model_CRNN_EfficientNetB0_BiLSTM import Model

In [2]:
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

from dataset import avesDataset, SpectrogramAugment, TransformedSubset
from train import train_model


data_dir = "../dataset"
full_dataset = avesDataset(data_dir)

# Pega o número real de classes do dataset
num_classes = len(full_dataset.label_map)
print(f"[INFO] Classes detectadas: {num_classes}")

# Instancia o modelo com o número correto de classes
model = Model(num_classes=num_classes)

# ──────────────────────────────────────────────────────────────────────────────
# Tópico 2: Split estratificado por label (evita data leakage por classe e
# garante que cada fold tenha a mesma proporção de cada espécie).
# ──────────────────────────────────────────────────────────────────────────────
all_indices = list(range(len(full_dataset)))
all_labels  = [full_dataset.samples[i][1] for i in all_indices]

# 80% treino | 20% temp
train_idx, temp_idx, _, temp_labels = train_test_split(
    all_indices, all_labels,
    test_size=0.2,
    stratify=all_labels,
    random_state=42
)

# 50% do temp → val  |  50% do temp → test  (= 10% / 10% do total)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42
)

print(f"[INFO] Split estratificado — treino: {len(train_idx)} | val: {len(val_idx)} | teste: {len(test_idx)}")

train_subset = Subset(full_dataset, train_idx)
val_subset   = Subset(full_dataset, val_idx)
test_subset  = Subset(full_dataset, test_idx)

# Augmentation apenas no treino
train_dataset = TransformedSubset(train_subset, transform=SpectrogramAugment())
val_dataset   = val_subset
test_dataset  = test_subset

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

train_model(
    model=model,
    num_epochs=50,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    patience=10,
    min_delta=0.0001,
    lr=0.0001
)


c:\Users\Miguel\anaconda3\envs\aves\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Classes detectadas: 27
[INFO] Split estratificado — treino: 8635 | val: 1079 | teste: 1080
Usando dispositivo: cuda
[INFO] Detectadas 27 classes dinamicamente.


c:\Users\Miguel\anaconda3\envs\aves\lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
c:\Users\Miguel\anaconda3\envs\aves\lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as dou

[INFO] Dataset 'dataset_4ad1c1b' registrado nativamente no MLflow com 10794 amostras.
Epoca [1/50] - Loss Treino: 2.2780, Acc Treino: 43.95% | Loss Val: 1.3761, Acc Val: 74.79%
[OK] Melhor modelo atualizado!
Epoca [2/50] - Loss Treino: 1.4691, Acc Treino: 72.45% | Loss Val: 1.2176, Acc Val: 80.63%
[OK] Melhor modelo atualizado!
Epoca [3/50] - Loss Treino: 1.3124, Acc Treino: 77.04% | Loss Val: 1.1807, Acc Val: 82.58%
[OK] Melhor modelo atualizado!
Epoca [4/50] - Loss Treino: 1.2331, Acc Treino: 80.17% | Loss Val: 1.1534, Acc Val: 83.41%
[OK] Melhor modelo atualizado!
Epoca [5/50] - Loss Treino: 1.1604, Acc Treino: 82.41% | Loss Val: 1.1177, Acc Val: 84.24%
[OK] Melhor modelo atualizado!
Epoca [6/50] - Loss Treino: 1.1113, Acc Treino: 84.13% | Loss Val: 1.1264, Acc Val: 84.06%
[WAIT] Sem melhora por 1 épocas
Epoca [7/50] - Loss Treino: 1.0646, Acc Treino: 85.58% | Loss Val: 1.1110, Acc Val: 84.62%
[OK] Melhor modelo atualizado!
Epoca [8/50] - Loss Treino: 1.0536, Acc Treino: 86.18% | Lo

AttributeError: 'Sequential' object has no attribute 'out_features'